## Cell 1

In [ ]:
!pip -q install "mcp[cli]"

## Cell 2

In [ ]:
import asyncio
import json
import os
import sqlite3

from contextlib import AsyncExitStack

print("Environment initialised successfully!")
print("MCP Business Operations System")

## Cell 3

In [ ]:
inventory_db = sqlite3.connect(
    "inventory.db",
    check_same_thread=False
)

inventory_db.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    reorder_level INTEGER NOT NULL
)
""")

inventory_db.commit()

print("Inventory database created successfully!")

## Cell 4

In [ ]:
products = [
    (1, "Laptop", 25, 65000, 5),
    (2, "Keyboard", 80, 1200, 15),
    (3, "Mouse", 12, 600, 20),
    (4, "Monitor", 18, 12000, 5),
    (5, "Headphones", 7, 2500, 10)
]

inventory_db.executemany("""
INSERT OR IGNORE INTO products
VALUES (?, ?, ?, ?, ?)
""", products)

inventory_db.commit()

print("Sample inventory inserted successfully!")

## Cell 5

In [ ]:
from mcp.server.mcpserver import MCPServer

inventory_server = MCPServer("Inventory Server")

@inventory_server.tool()
def get_inventory():
    """Retrieve all products and their stock levels."""

    cursor = inventory_db.execute(
        "SELECT * FROM products"
    )

    return [
        {
            "product_id": row[0],
            "name": row[1],
            "quantity": row[2],
            "unit_price": row[3],
            "reorder_level": row[4]
        }
        for row in cursor.fetchall()
    ]


@inventory_server.tool()
def check_stock(product_id: int):
    """Check the stock level of a particular product."""

    cursor = inventory_db.execute(
        "SELECT * FROM products WHERE product_id = ?",
        (product_id,)
    )

    row = cursor.fetchone()

    if row is None:
        return {"error": "Product not found"}

    return {
        "product_id": row[0],
        "name": row[1],
        "quantity": row[2],
        "unit_price": row[3],
        "reorder_level": row[4]
    }


@inventory_server.tool()
def add_stock(product_id: int, quantity: int):
    """Add stock to an existing product."""

    if quantity <= 0:
        return {"error": "Quantity must be positive"}

    cursor = inventory_db.execute(
        "UPDATE products SET quantity = quantity + ? "
        "WHERE product_id = ?",
        (quantity, product_id)
    )

    inventory_db.commit()

    if cursor.rowcount == 0:
        return {"error": "Product not found"}

    return {
        "message": "Stock updated successfully",
        "product_id": product_id,
        "added_quantity": quantity
    }


@inventory_server.tool()
def get_low_stock():
    """Identify products requiring stock replenishment."""

    cursor = inventory_db.execute("""
        SELECT name, quantity, reorder_level
        FROM products
        WHERE quantity <= reorder_level
    """)

    return [
        {
            "name": row[0],
            "quantity": row[1],
            "reorder_level": row[2]
        }
        for row in cursor.fetchall()
    ]


print("Inventory MCP Server created successfully!")
print("Registered tools: 4")

## Cell 6

In [ ]:
print("STOCK CHECK")
print(check_stock(3))

## Cell 7

In [ ]:
finance_db = sqlite3.connect(
    "finance.db",
    check_same_thread=False
)

finance_db.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id INTEGER PRIMARY KEY,
    description TEXT NOT NULL,
    amount REAL NOT NULL,
    transaction_type TEXT NOT NULL
)
""")

finance_db.commit()

print("Financial database created successfully!")

## Cell 8

In [ ]:
transactions = [
    (1, "Laptop Sales", 195000, "income"),
    (2, "Keyboard Sales", 24000, "income"),
    (3, "Office Rent", 30000, "expense"),
    (4, "Employee Salaries", 45000, "expense"),
    (5, "Electricity Bill", 5000, "expense"),
    (6, "Mouse Sales", 12000, "income")
]

finance_db.executemany("""
INSERT OR IGNORE INTO transactions
VALUES (?, ?, ?, ?)
""", transactions)

finance_db.commit()

print("Sample financial transactions inserted successfully!")

## Cell 9

In [ ]:
finance_server = MCPServer("Finance Server")

@finance_server.tool()
def get_transactions():
    """Retrieve all financial transactions."""

    cursor = finance_db.execute(
        "SELECT * FROM transactions"
    )

    rows = cursor.fetchall()

    return [
        {
            "transaction_id": row[0],
            "description": row[1],
            "amount": row[2],
            "transaction_type": row[3]
        }
        for row in rows
    ]


@finance_server.tool()
def calculate_financial_summary():
    """Calculate total income, expenses and profit."""

    cursor = finance_db.execute("""
        SELECT
            SUM(CASE WHEN transaction_type = 'income'
                THEN amount ELSE 0 END),
            SUM(CASE WHEN transaction_type = 'expense'
                THEN amount ELSE 0 END)
        FROM transactions
    """)

    income, expenses = cursor.fetchone()

    income = income or 0
    expenses = expenses or 0

    return {
        "total_income": income,
        "total_expenses": expenses,
        "net_profit": income - expenses
    }

print("Finance MCP server created successfully!")

## Cell 10

In [ ]:
print("FINANCIAL TRANSACTIONS")
print(get_transactions())

print("\nFINANCIAL SUMMARY")
print(calculate_financial_summary())

## Cell 11

In [ ]:
sales_db = sqlite3.connect(
    "sales.db",
    check_same_thread=False
)

sales_db.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    product_name TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    total_amount REAL NOT NULL,
    status TEXT NOT NULL
)
""")

sales_db.commit()

print("Sales database created successfully!")

## Cell 12

In [ ]:
orders = [
    (1, "Aarav", "Laptop", 2, 130000, "completed"),
    (2, "Mira", "Keyboard", 5, 6000, "pending"),
    (3, "Rohan", "Mouse", 3, 1800, "processing"),
    (4, "Anaya", "Monitor", 1, 12000, "completed"),
    (5, "Ishaan", "Headphones", 2, 5000, "pending")
]

sales_db.executemany("""
INSERT OR IGNORE INTO orders
VALUES (?, ?, ?, ?, ?, ?)
""", orders)

sales_db.commit()

print("Sample customer orders inserted successfully!")

## Cell 13

In [ ]:
sales_server = MCPServer("Sales and Orders Server")


@sales_server.tool()
def get_orders():
    """Retrieve all customer orders."""

    cursor = sales_db.execute(
        "SELECT * FROM orders"
    )

    rows = cursor.fetchall()

    return [
        {
            "order_id": row[0],
            "customer_name": row[1],
            "product_name": row[2],
            "quantity": row[3],
            "total_amount": row[4],
            "status": row[5]
        }
        for row in rows
    ]


@sales_server.tool()
def get_pending_orders():
    """Retrieve all orders awaiting completion."""

    cursor = sales_db.execute(
        "SELECT * FROM orders WHERE status = 'pending'"
    )

    rows = cursor.fetchall()

    return [
        {
            "order_id": row[0],
            "customer_name": row[1],
            "product_name": row[2],
            "quantity": row[3],
            "total_amount": row[4],
            "status": row[5]
        }
        for row in rows
    ]


@sales_server.tool()
def calculate_total_sales():
    """Calculate the total value of all orders."""

    cursor = sales_db.execute(
        "SELECT SUM(total_amount) FROM orders"
    )

    return cursor.fetchone()[0] or 0


print("Sales and Orders MCP server created successfully!")

## Cell 14

In [ ]:
print("ALL CUSTOMER ORDERS")
print(get_orders())

print("\nPENDING ORDERS")
print(get_pending_orders())

print("\nTOTAL SALES")
print(calculate_total_sales())

## Cell 15

In [ ]:
import inspect

print("MCP SERVER TYPE:")
print(type(inventory_server))

print("\nAVAILABLE SERVER METHODS:")
print([
    method for method in dir(inventory_server)
    if not method.startswith("_")
])

print("\nSERVER INITIALISATION:")
print(inspect.signature(MCPServer))

## Cell 16

In [ ]:
import mcp.client
import pkgutil

print("AVAILABLE MCP CLIENT MODULES:")

print([
    module.name
    for module in pkgutil.iter_modules(mcp.client.__path__)
])

## Cell 17

In [ ]:
import inspect

from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client

print("CLIENT SESSION:")
print(inspect.signature(ClientSession))

print("\nSTDIO CLIENT:")
print(inspect.signature(stdio_client))

## Cell 18

In [ ]:
import inspect

from mcp.client.streamable_http import streamable_http_client

print("STREAMABLE HTTP CLIENT:")
print(inspect.signature(streamable_http_client))

print("\nSERVER HTTP RUNNER:")
print(inspect.signature(inventory_server.run_streamable_http_async))

## Cell 19

In [ ]:
import inspect

print(inspect.getsource(
    MCPServer.run_streamable_http_async
))

## Cell 20

In [ ]:
server_tasks = {
    "Inventory": asyncio.create_task(
        inventory_server.run_streamable_http_async(
            host="127.0.0.1",
            port=8001
        )
    ),

    "Finance": asyncio.create_task(
        finance_server.run_streamable_http_async(
            host="127.0.0.1",
            port=8002
        )
    ),

    "Sales": asyncio.create_task(
        sales_server.run_streamable_http_async(
            host="127.0.0.1",
            port=8003
        )
    )
}

await asyncio.sleep(3)

for name, task in server_tasks.items():
    print(
        name,
        "server:",
        "RUNNING" if not task.done() else "STOPPED"
    )

## Cell 21

In [ ]:
import inspect

print(inspect.getsource(streamable_http_client))

print("\nCLIENT SESSION METHODS:")

print([
    method for method in dir(ClientSession)
    if not method.startswith("_")
])

## Cell 22

In [ ]:
import inspect

print("INITIALIZE:")
print(inspect.signature(ClientSession.initialize))

print("\nLIST TOOLS:")
print(inspect.signature(ClientSession.list_tools))

print("\nCALL TOOL:")
print(inspect.signature(ClientSession.call_tool))

print("\nSESSION CONTEXT MANAGER:")
print(hasattr(ClientSession, "__aenter__"))
print(hasattr(ClientSession, "__aexit__"))

## Cell 23

In [ ]:
from mcp.client.session import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def inspect_server(name, url):

    async with streamable_http_client(url) as streams:

        read_stream, write_stream = streams

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            tools = await session.list_tools()

            print(f"\n{name.upper()} SERVER")
            print("Connected successfully!")

            for tool in tools.tools:
                print("-", tool.name)


await inspect_server(
    "Inventory",
    "http://127.0.0.1:8001/mcp"
)

await inspect_server(
    "Finance",
    "http://127.0.0.1:8002/mcp"
)

await inspect_server(
    "Sales",
    "http://127.0.0.1:8003/mcp"
)

## Cell 24

In [ ]:
client_stack = AsyncExitStack()
await client_stack.__aenter__()

if "client_sessions" not in globals():
    client_sessions = {}

async def connect_server(name, url):

    streams = await client_stack.enter_async_context(
        streamable_http_client(url)
    )

    session = await client_stack.enter_async_context(
        ClientSession(*streams)
    )

    await session.initialize()

    client_sessions[name] = session

    print(f"{name} connected and session established!")


await connect_server(
    "inventory",
    "http://127.0.0.1:8001/mcp"
)

await connect_server(
    "finance",
    "http://127.0.0.1:8002/mcp"
)

await connect_server(
    "sales",
    "http://127.0.0.1:8003/mcp"
)

## Cell 25

In [ ]:
tool_registry = {}

for server_name, session in client_sessions.items():

    result = await session.list_tools()

    for tool in result.tools:
        tool_registry[tool.name] = {
            "server": server_name,
            "description": tool.description
        }


async def call_mcp_tool(tool_name, arguments=None):

    if tool_name not in tool_registry:
        return {"error": "Tool not found"}

    server_name = tool_registry[tool_name]["server"]

    session = client_sessions[server_name]

    result = await session.call_tool(
        tool_name,
        arguments or {}
    )

    return result


print("CENTRAL MCP TOOL REGISTRY")

for name, details in tool_registry.items():
    print(f"{name} -> {details['server']}")

print("\nTotal registered tools:", len(tool_registry))

## Cell 26

In [ ]:
print("INVENTORY")
inventory_result = await call_mcp_tool("get_inventory")
print(inventory_result)

print("\nFINANCIAL SUMMARY")
finance_result = await call_mcp_tool(
    "calculate_financial_summary"
)
print(finance_result)

print("\nPENDING ORDERS")
orders_result = await call_mcp_tool("get_pending_orders")
print(orders_result)

## Cell 27

In [ ]:
def normalize_mcp_result(result):

    if result.is_error:
        return {
            "error": [
                item.text
                for item in result.content
                if hasattr(item, "text")
            ]
        }

    if result.structured_content is not None:
        return result.structured_content

    values = []

    for item in result.content:
        if hasattr(item, "text"):
            try:
                values.append(json.loads(item.text))
            except json.JSONDecodeError:
                values.append(item.text)

    if len(values) == 1:
        return values[0]

    return values


async def call_mcp_tool(tool_name, arguments=None):

    if tool_name not in tool_registry:
        return {"error": "Tool not found"}

    server_name = tool_registry[tool_name]["server"]

    session = client_sessions[server_name]

    result = await session.call_tool(
        tool_name,
        arguments or {}
    )

    return normalize_mcp_result(result)


print("MCP result normalization ready!")

## Cell 28

In [ ]:
print("INVENTORY")
print(await call_mcp_tool("get_inventory"))

print("\nFINANCIAL SUMMARY")
print(await call_mcp_tool("calculate_financial_summary"))

print("\nPENDING ORDERS")
print(await call_mcp_tool("get_pending_orders"))

## Cell 29

In [ ]:
async def get_business_snapshot():

    inventory, finance, orders = await asyncio.gather(
        call_mcp_tool("get_inventory"),
        call_mcp_tool("calculate_financial_summary"),
        call_mcp_tool("get_orders")
    )

    return {
        "inventory": inventory,
        "finance": finance,
        "orders": orders
    }


business_snapshot = await get_business_snapshot()

print("Unified business snapshot created!")

print("\nFINANCIAL SUMMARY:")
print(business_snapshot["finance"])

print("\nPRODUCT COUNT:")
print(len(business_snapshot["inventory"]))

print("\nORDER COUNT:")
print(len(business_snapshot["orders"]))

## Cell 30

In [ ]:
async def monitor_business():

    snapshot = await get_business_snapshot()

    alerts = []

    for product in snapshot["inventory"]:
        if product["quantity"] <= product["reorder_level"]:
            alerts.append({
                "type": "LOW STOCK",
                "message": (
                    f"{product['name']} has only "
                    f"{product['quantity']} units remaining."
                )
            })

    for order in snapshot["orders"]:
        if order["status"] == "pending":
            alerts.append({
                "type": "PENDING ORDER",
                "message": (
                    f"Order {order['order_id']} for "
                    f"{order['customer_name']} is pending."
                )
            })

    if snapshot["finance"]["net_profit"] < 0:
        alerts.append({
            "type": "FINANCIAL WARNING",
            "message": "The business is operating at a loss."
        })

    return alerts


business_alerts = await monitor_business()

print("BUSINESS MONITORING REPORT")

for alert in business_alerts:
    print(f"\n[{alert['type']}] {alert['message']}")

## Cell 31

In [ ]:
pending_approvals = {}
approval_counter = 0

WRITE_TOOLS = {
    "add_stock"
}


def request_approval(tool_name, arguments):

    global approval_counter

    if tool_name not in WRITE_TOOLS:
        return {
            "error": "This operation does not require approval."
        }

    approval_counter += 1

    action_id = approval_counter

    pending_approvals[action_id] = {
        "tool": tool_name,
        "arguments": arguments,
        "status": "pending"
    }

    return {
        "action_id": action_id,
        "tool": tool_name,
        "arguments": arguments,
        "status": "Awaiting approval"
    }


print("Permission and approval system created!")

## Cell 32

In [ ]:
def approve_action(action_id):

    if action_id not in pending_approvals:
        return {"error": "Approval request not found"}

    action = pending_approvals[action_id]

    if action["status"] != "pending":
        return {"error": "Request already processed"}

    action["status"] = "approved"

    return {
        "action_id": action_id,
        "status": "approved"
    }


def reject_action(action_id):

    if action_id not in pending_approvals:
        return {"error": "Approval request not found"}

    action = pending_approvals[action_id]

    if action["status"] != "pending":
        return {"error": "Request already processed"}

    action["status"] = "rejected"

    return {
        "action_id": action_id,
        "status": "rejected"
    }


print("Approval and rejection mechanisms created!")

## Cell 33

In [ ]:
for tool in inventory_tools.tools:
    if tool.name == "add_stock":
        print("TOOL DETAILS:")
        print(json.dumps(tool.model_dump(), indent=4))

## Cell 34

In [ ]:
async def _call_mcp_tool(tool_name, arguments=None):

    server_name = tool_registry[tool_name]["server"]
    session = client_sessions[server_name]

    result = await session.call_tool(tool_name, arguments or {})
    return normalize_mcp_result(result)


async def call_mcp_tool(tool_name, arguments=None):

    if tool_name in WRITE_TOOLS:
        return {
            "error": "Approval required before executing this operation."
        }

    return await _call_mcp_tool(tool_name, arguments)


async def execute_approved_action(action_id):

    if action_id not in pending_approvals:
        return {"error": "Approval request not found"}

    action = pending_approvals[action_id]

    if action["status"] != "approved":
        return {"error": "Action has not been approved"}

    result = await _call_mcp_tool(
        action["tool"],
        action["arguments"]
    )

    action["result"] = result
    action["status"] = "executed"

    return {
        "action_id": action_id,
        "status": "executed",
        "result": result
    }


print("Approval-controlled execution system created!")

## Cell 35

In [ ]:
approval_request = request_approval(
    "add_stock",
    {
        "product_id": 3,
        "quantity": 1
    }
)

print(json.dumps(approval_request, indent=4))

## Cell 36

In [ ]:
approval_result = approve_action(1)

print(json.dumps(approval_result, indent=4))

## Cell 37

In [ ]:
execution_result = await execute_approved_action(1)

print(json.dumps(execution_result, indent=4))

## Cell 38

In [ ]:
mouse_stock = await call_mcp_tool(
    "check_stock",
    {"product_id": 3}
)

print(json.dumps(mouse_stock, indent=4))

## Cell 39

In [ ]:
blocked_result = await call_mcp_tool(
    "add_stock",
    {
        "product_id": 3,
        "quantity": 5
    }
)

print(json.dumps(blocked_result, indent=4))

## Cell 40

In [ ]:
import os

print("File created:", os.path.exists('/content/my_notebook.ipynb'))

if os.path.exists('/content/my_notebook.ipynb'):
    print("File size:", os.path.getsize('/content/my_notebook.ipynb'), "bytes")
else:
    print("Notebook file was not saved.")